# Lab 9 — Local Ollama Chatbot (Windows 11 / llama3.2:3b)

## Learning Goals

- Run a capable open-source LLM locally on Windows 11
- Implement Ollama in a Python application
- Manage conversation history for multi-turn chat
- Understand why OS overhead matters for local AI

---
### ✅ Prerequisites Already Complete

Ollama and all system dependencies are already installed. On Windows, Ollama runs automatically as a background service after installation — no need to run `ollama serve` manually.

---
## Step 1 — Verify System Dependencies

In [1]:
import subprocess, sys, shutil

print(f"Python version: {sys.version}\n")

tools = ["python", "pip", "ollama"]
missing = [t for t in tools if shutil.which(t) is None]

if missing:
    print(f"⚠️  Missing tools: {missing}")
    print("Please install them before continuing:")
    print("  Python: https://www.python.org/downloads/")
    print("  Ollama: https://ollama.com/download/windows")
else:
    print("✅ All system dependencies are present.")

Python version: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]

✅ All system dependencies are present.


---
## Step 2 — Verify Ollama is Running

In [2]:
import subprocess

# Check Ollama version
result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
print(f"Ollama version: {result.stdout.strip()}")

# Check if Ollama API is reachable (auto-starts on Windows after install)
import urllib.request
try:
    urllib.request.urlopen("http://127.0.0.1:11434", timeout=3)
    print("✅ Ollama service is running.")
except Exception:
    print("⚠️  Ollama service not detected.")
    print("   Try launching Ollama from the Start menu, or run in PowerShell:")
    print("   ollama serve")

Ollama version: ollama version is 0.17.7
✅ Ollama service is running.


---
## Step 3 — Verify Model is Available

We use `llama3.2:3b` — a fast, capable model that fits in ~2 GB RAM, ideal for Windows 11.

If the model is missing, pull it in PowerShell:
```powershell
ollama pull llama3.2:3b
```

In [3]:
import subprocess

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)

if "llama3.2:3b" in result.stdout:
    print("✅ llama3.2:3b is available.")
else:
    print("⚠️  llama3.2:3b not found. Run in PowerShell:")
    print("   ollama pull llama3.2:3b")

NAME           ID              SIZE      MODIFIED     
llama3.2:3b    a80c4f17acd5    2.0 GB    2 hours ago     
llama3.1:8b    46e0c10c039e    4.9 GB    19 hours ago    

✅ llama3.1:8b is available.


---
## Step 4 — Set Up Python Virtual Environment

Creates a venv at `%USERPROFILE%\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3\venv` and installs the `ollama` Python package. Safe to re-run — skips creation if the venv already exists.

In [4]:
import subprocess, os, sys

venv_path   = os.path.join(os.path.expanduser("~"), "Documents", "GitHub", "CSB430SWIWinter2026", "Labs", "Lab9", "llama3", "venv")
pip_path    = os.path.join(venv_path, "Scripts", "pip.exe")
python_path = os.path.join(venv_path, "Scripts", "python.exe")

# Ensure the parent directory exists
os.makedirs(os.path.dirname(venv_path), exist_ok=True)

# Create venv if it doesn't exist
if not os.path.exists(python_path):
    print("Creating virtual environment...")
    subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
    print("✅ venv created at", venv_path)
else:
    print("✅ venv already exists at", venv_path)

# Upgrade pip
print("Upgrading pip...")
subprocess.run([python_path, "-m", "pip", "install", "--upgrade", "pip"],
               capture_output=True)

# Install ollama if missing
check = subprocess.run([python_path, "-c", "import ollama; print(ollama.__version__)"],
                       capture_output=True, text=True)
if check.returncode == 0:
    print("✅ ollama package already installed:", check.stdout.strip())
else:
    print("Installing ollama package...")
    subprocess.run([pip_path, "install", "ollama"], check=True)
    print("✅ ollama installed.")

✅ venv found at C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/venv
⚠️  ollama package missing. Run: C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/venv/bin/pip install ollama


---
## Step 5 — Check Available RAM

You want at least **2.5 GB free** for `llama3.2:3b` to run comfortably.

In [5]:
import subprocess

result = subprocess.run(
    ["wmic", "OS", "get", "FreePhysicalMemory", "/Value"],
    capture_output=True, text=True
)

free_kb = None
for line in result.stdout.splitlines():
    if "FreePhysicalMemory" in line and "=" in line:
        free_kb = int(line.split("=")[1].strip())
        break

if free_kb is None:
    print("⚠️  Could not read available RAM.")
else:
    gb = free_kb / (1024 ** 2)
    print(f"Available RAM: {gb:.1f} GB free")
    if gb < 2.5:
        print("⚠️  Less than 2.5 GB free. Close other applications before running the model.")
    else:
        print("✅ RAM looks good — safe to run llama3.2:3b.")

               total        used        free      shared  buff/cache   available
Mem:           6.7Gi       2.6Gi       2.1Gi        47Mi       1.9Gi       3.7Gi
Swap:          2.0Gi       882Mi       1.1Gi

⚠️  Only 3.7Gi free. Close other applications before running the model.


---
## Step 6 — Test Ollama in Python

Verify the model responds from inside Python before building the full chatbot.

In [10]:
import subprocess, os

python_path = os.path.join(os.path.expanduser("~"), "Documents", "GitHub", "CSB430SWIWinter2026", "Labs", "Lab9", "llama3", "venv", "Scripts", "python.exe")

test_script = """
from ollama import chat
response = chat(
    model='llama3.2:3b',
    messages=[{'role': 'user', 'content': 'Hello! Respond in one sentence.'}],
)
print(response.message.content)
"""

result = subprocess.run([python_path, "-c", test_script], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ Model response:")
    print(result.stdout)
else:
    print("❌ Error:")
    print(result.stderr)

❌ Error:
Traceback (most recent call last):
  File "<string>", line 3, in <module>
  File "C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/venv/lib/python3.10/site-packages/ollama/_client.py", line 365, in chat
    return self._request(
  File "C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/venv/lib/python3.10/site-packages/ollama/_client.py", line 189, in _request
    return cls(**self._request_raw(*args, **kwargs).json())
  File "C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/venv/lib/python3.10/site-packages/ollama/_client.py", line 133, in _request_raw
    raise ResponseError(e.response.text, e.response.status_code) from None
ollama._types.ResponseError: model requires more system memory (4.8 GiB) than is available (4.7 GiB) (status code: 500)



---
## Step 7 — Understanding Conversation History

Ollama is **stateless** — each API call knows nothing about previous ones. To give the chatbot memory, we pass the full conversation history with every request:

```
[ golem.py ]
     |
     |  chat(model, messages=[...full history...])
     ↓
[ Ollama: llama3.2:3b ]
     |
     |  response
     ↓
[ golem.py ] --> appends reply to history --> loops
```

Each message has a `role` (`user` or `assistant`) and `content`. Both sides of every exchange are appended so the model can refer back to earlier context.

---
## Step 8 — Write the Golem Chatbot (golem.py)

In [11]:
import os

chat_code = '''\
import tkinter as tk
import threading
from ollama import chat

history = []

def send_message(event=None):
    user_input = entry.get().strip()
    if not user_input:
        return
    entry.delete(0, tk.END)
    append_message("You", user_input)
    history.append({"role": "user", "content": user_input})
    send_btn.config(state=tk.DISABLED)
    entry.config(state=tk.DISABLED)
    threading.Thread(target=get_reply, daemon=True).start()

def get_reply():
    try:
        response = chat(model="llama3.2:3b", messages=history)
        reply = response.message.content
        history.append({"role": "assistant", "content": reply})
        root.after(0, lambda: append_message("Golem", reply))
    except Exception as e:
        root.after(0, lambda: append_message("Error", str(e)))
    finally:
        root.after(0, re_enable)

def re_enable():
    send_btn.config(state=tk.NORMAL)
    entry.config(state=tk.NORMAL)
    entry.focus()

def append_message(sender, text):
    chat_box.config(state=tk.NORMAL)
    chat_box.insert(tk.END, sender + ":" + chr(10) + text + chr(10) * 2)
    chat_box.config(state=tk.DISABLED)
    chat_box.see(tk.END)

root = tk.Tk()
root.title("Golem")
root.geometry("600x500")
root.resizable(True, True)

chat_box = tk.Text(root, state=tk.DISABLED, wrap=tk.WORD, font=("Segoe UI", 11),
                   bg="#1e1e1e", fg="#e0e0e0", padx=8, pady=8, relief=tk.FLAT)
chat_box.pack(fill=tk.BOTH, expand=True, padx=8, pady=(8, 4))

frame = tk.Frame(root, bg="#2b2b2b")
frame.pack(fill=tk.X, padx=8, pady=(0, 8))

entry = tk.Entry(frame, font=("Segoe UI", 11), bg="#2b2b2b", fg="#e0e0e0",
                 insertbackground="white", relief=tk.FLAT)
entry.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=6, padx=(0, 6))
entry.bind("<Return>", send_message)
entry.focus()

send_btn = tk.Button(frame, text="Send", command=send_message,
                     bg="#444", fg="white", relief=tk.FLAT,
                     font=("Segoe UI", 11), padx=12)
send_btn.pack(side=tk.RIGHT)

append_message("Golem", "Hello! I am Golem. How can I help you today?")

root.mainloop()
'''

llm_dir   = os.path.join(os.path.expanduser("~"), "Documents", "GitHub", "CSB430SWIWinter2026", "Labs", "Lab9", "llama3")
os.makedirs(llm_dir, exist_ok=True)
chat_path = os.path.join(llm_dir, "golem.py")
with open(chat_path, 'w') as f:
    f.write(chat_code)

print(f"✅ golem.py written to {chat_path}")

✅ golem.py written to C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/golem.py


---
## Step 9 — Run the Chatbot

Launch the GUI from PowerShell:

```powershell
cd ~\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3
.\venv\Scripts\activate
python golem.py
```

A dark chat window will open. Type your message and press **Enter** or **Send**.

> **Note:** The first response may take 10–15 seconds while the model loads into RAM.

---
## Step 10 — Create the Desktop Launcher

This writes a `launch_golem.bat` batch file and creates a proper Windows shortcut (`Golem.lnk`) on your Desktop. Double-clicking the shortcut launches Golem with no terminal window appearing.

In [14]:
import os, subprocess

home        = os.path.expanduser("~")
llm_dir     = os.path.join(home, "Documents", "GitHub", "CSB430SWIWinter2026", "Labs", "Lab9", "llama3")
venv_python = os.path.join(llm_dir, "venv", "Scripts", "pythonw.exe")  # pythonw = no console window
golem_py    = os.path.join(llm_dir, "golem.py")
bat_path    = os.path.join(llm_dir, "launch_golem.bat")
shortcut    = os.path.join(home, "Desktop", "Golem.lnk")

# Batch file: start Golem silently (pythonw suppresses the console)
bat_code = f'''@echo off
start "" "{venv_python}" "{golem_py}"
'''
with open(bat_path, 'w') as f:
    f.write(bat_code)
print(f"✅ launch_golem.bat written to {bat_path}")

# Create a proper .lnk shortcut via PowerShell
ps_script = f"""
$ws  = New-Object -ComObject WScript.Shell
$lnk = $ws.CreateShortcut('{shortcut}')
$lnk.TargetPath      = '{bat_path}'
$lnk.WorkingDirectory = '{llm_dir}'
$lnk.WindowStyle     = 7  # minimized — hides the brief cmd flash
$lnk.Description     = 'Golem — Local AI Chatbot'
$lnk.Save()
"""
result = subprocess.run(["powershell", "-Command", ps_script], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ Golem.lnk shortcut created on Desktop")
    print()
    print("👉 Double-click 'Golem' on your Desktop to launch the chatbot.")
else:
    print("⚠️  Shortcut creation failed:")
    print(result.stderr)

✅ launch_golem.sh written to C:\Users\flori\Documents\GitHub\CSB430SWIWinter2026\Labs\Lab9\llama3/launch_golem.sh
✅ Golem.desktop written to /home/flori/Desktop/Golem.desktop

👉 On your desktop, right-click Golem.desktop → 'Allow Launching' if prompted.


---
## Step 11 — Verify All Files

In [15]:
import os

home = os.path.expanduser("~")
base = os.path.join(home, "Documents", "GitHub", "CSB430SWIWinter2026", "Labs", "Lab9", "llama3")

files = {
    os.path.join(base, "golem.py"):              "Main chatbot GUI",
    os.path.join(base, "launch_golem.bat"):      "Launcher batch file",
    os.path.join(home, "Desktop", "Golem.lnk"): "Desktop shortcut",
    os.path.join(base, "requirements.txt"):      "Python dependencies",
    os.path.join(base, "venv"):                  "Virtual environment",
}

for full, desc in files.items():
    status = "✅ EXISTS" if os.path.exists(full) else "❌ MISSING"
    print(f"{status}  —  {os.path.basename(full)} ({desc})")

✅ EXISTS  —  golem.py (Main chatbot GUI)
✅ EXISTS  —  launch_golem.sh (Launcher wrapper script)
✅ EXISTS  —  Golem.desktop (Desktop icon)
✅ EXISTS  —  requirements.txt (Python dependencies)
✅ EXISTS  —  venv (Virtual environment)
